# SCRAM - L2 observer fine-tune (QLoRA on Qwen2.5-1.5B)

Trains the **L2 observer** (guardian): a 6-label classifier that reads either an
operator **proposal** (setpoints + rationale + recent setpoints) or a 30 s
**telemetry window** and answers `{"label", "score", "reason"}`.

Labels: `normal`, `excursion`, `sensor_fault`, `actuator_fault`,
`rationale_mismatch`, `setpoint_drift`.

**Before you run anything:** Runtime -> Change runtime type -> **T4 GPU**.

Data: `observer_train.jsonl`, `observer_val.jsonl`, `observer_unseen.jsonl` from
`training/` (built by `python -m training.build_observer_dataset`). Every example
was formatted by `monitor/observer_prompt_format.py`, the same module that serves
the model, so training matches serving.

## 0. Install

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps "xformers<0.0.27" "trl<0.12.0" peft accelerate bitsandbytes

## 1. Load the base model (4-bit)
Same base as the operator. 4-bit weights keep this well inside a T4's 16 GB (v2
was trained in fp16 and peaked at ~10 GB). Telemetry prompts are the longest
(~1k tokens with the system prompt), so 2048 is ample.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL,
    max_seq_length = MAX_SEQ,
    load_in_4bit   = True,
    dtype          = None,   # auto: fp16 on a T4
)

## 2. Attach the LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
)
model.print_trainable_parameters()

## 3. Upload and format the data
Pick `observer_train.jsonl`, `observer_val.jsonl` and `observer_unseen.jsonl`.
Each line is `{"messages":[system, user, assistant]}`.

In [ ]:
from google.colab import files
up = files.upload()   # choose the three observer_*.jsonl files

from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)}

train_ds  = load_dataset("json", data_files="observer_train.jsonl",  split="train").map(to_text)
val_ds    = load_dataset("json", data_files="observer_val.jsonl",    split="train").map(to_text)
unseen_ds = load_dataset("json", data_files="observer_unseen.jsonl", split="train")
print(train_ds, val_ds, unseen_ds)
lens = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"][:300]]
print("token length (first 300): max", max(lens), "mean", sum(lens) // len(lens))
print("\n--- one formatted example ---\n")
print(train_ds[0]["text"][:1500])

## 4. Train (1 epoch, responses only)
One epoch like v2: the classes are templated, so more passes mostly memorise.
`train_on_responses_only` puts the loss on the assistant JSON only.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 10,
        num_train_epochs            = 1,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        eval_strategy               = "steps",
        eval_steps                  = 150,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

stats = trainer.train()
print(stats)

## 5. Evaluate on val AND unseen
Same report shape as v2's `eval_observer_val.json` so the numbers compare:
accuracy, valid-JSON rate, per-class recall, detection rate (attack classes
flagged as any attack), false-alarm rate (normal flagged), confusion, and the
same metrics split by input type. `EVAL_N = None` runs every example (~30-40 min
on a T4); set e.g. 300 for a quick look.

In [ ]:
import json, random
from collections import Counter, defaultdict
FastLanguageModel.for_inference(model)

LABELS = ["normal","excursion","sensor_fault","actuator_fault","rationale_mismatch","setpoint_drift"]
EVAL_N = None

def ask(messages_wo_answer):
    prompt = tokenizer.apply_chat_template(messages_wo_answer, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=96, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

def parse(text):   # same rules as monitor/observer_prompt_format.parse
    s, e = text.find("{"), text.rfind("}")
    if s < 0 or e <= s: return None
    try: obj = json.loads(text[s:e+1])
    except ValueError: return None
    return obj.get("label") if isinstance(obj, dict) and obj.get("label") in LABELS else None

def metrics(rows):
    if not rows: return None
    n = len(rows); valid = sum(r["pred"] is not None for r in rows)
    gold = Counter(r["gold"] for r in rows)
    recall = {c: (sum(r["gold"] == c and r["pred"] == c for r in rows) / gold[c]) if gold[c] else None
              for c in LABELS}
    att = [r for r in rows if r["gold"] != "normal"]; nor = [r for r in rows if r["gold"] == "normal"]
    conf = defaultdict(Counter)
    for r in rows: conf[r["gold"]][r["pred"] or "invalid"] += 1
    return {"n": n, "valid_json": valid / n, "accuracy": sum(r["gold"] == r["pred"] for r in rows) / n,
            "per_class_recall": recall,
            "detection_rate": (sum(r["pred"] not in (None, "normal") for r in att) / len(att)) if att else None,
            "false_alarm_rate": (sum(r["pred"] not in (None, "normal") for r in nor) / len(nor)) if nor else None,
            "n_normal": len(nor), "confusion": {g: dict(c) for g, c in conf.items()}}

def evaluate(ds, split):
    idx = list(range(len(ds)))
    if EVAL_N:
        random.Random(0).shuffle(idx); idx = idx[:EVAL_N]
    rows = []
    for i in idx:
        msgs = ds[i]["messages"]
        raw = ask(msgs[:2])
        rows.append({"gold": json.loads(msgs[2]["content"])["label"], "pred": parse(raw), "raw": raw,
                     "kind": json.loads(msgs[1]["content"])["input"]})
    rep = {"split": split, **metrics(rows),
           "by_kind": {"proposals": metrics([r for r in rows if r["kind"] == "proposal"]),
                       "telemetry": metrics([r for r in rows if r["kind"] == "telemetry"])},
           "predictions": [{"gold": r["gold"], "pred": r["pred"], "raw": r["raw"]} for r in rows]}
    json.dump(rep, open(f"eval_{split}.json", "w"), indent=1)
    print(f"== {split}: n={rep['n']} acc={rep['accuracy']:.3f} valid={rep['valid_json']:.3f} "
          f"detect={rep['detection_rate']:.3f} false_alarm={rep['false_alarm_rate']:.3f}")
    for c in LABELS: print(f"   {c:20s} recall={rep['per_class_recall'][c]}")
    for k, m in rep["by_kind"].items():
        if m: print(f"   by input {k:10s} n={m['n']} acc={m['accuracy']:.3f}")
    return rep

rep_val = evaluate(val_ds, "observer_val")
rep_unseen = evaluate(unseen_ds, "observer_test_unseen")
files.download("eval_observer_val.json"); files.download("eval_observer_test_unseen.json")

## 6. Export GGUF for Ollama
Merges the LoRA adapter into the base and quantises to `q4_k_m`, then writes a
Modelfile with the observer system prompt taken from a training example, so
Ollama serves exactly what was trained.

In [ ]:
GGUF_DIR = "scram-observer-gguf"
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

import glob, os
gguf = os.path.basename(glob.glob(GGUF_DIR + "/*.gguf")[0])
sys_prompt = json.loads(open("observer_train.jsonl").readline())["messages"][0]["content"]
Q3 = chr(34) * 3
modelfile = ("FROM ./" + gguf + "\n"
             "PARAMETER temperature 0\n"
             "PARAMETER num_ctx 2048\n"
             "SYSTEM " + Q3 + sys_prompt + Q3 + "\n")
open(GGUF_DIR + "/Modelfile", "w").write(modelfile)
json.dump(stats.metrics, open(GGUF_DIR + "/train_metrics.json", "w"), indent=1)
!cp eval_observer_val.json eval_observer_test_unseen.json {GGUF_DIR}/
!cd {GGUF_DIR} && zip -r ../scram-observer.zip . > /dev/null
files.download("scram-observer.zip")

## 7. Back on your machine
```bash
mkdir -p model/scram-observer-gguf && cd model/scram-observer-gguf && unzip ../../scram-observer.zip
ollama create scram-observer -f Modelfile
```
Bring back the two eval reports and the train loss and compare with v2 (87.1%
val / 72.8% unseen - but v2's numbers included the trivial provenance_violation
class this dataset drops) before wiring it into the live engine.